# Generación de Lenguaje Natural con DistilGPT2
Este notebook guía paso a paso cómo utilizar un modelo preentrenado de generación de texto (DistilGPT2) para que puedas explicar cada estrategia a tus alumnos. Ideal para el curso de NLG.

## Paso 1: Instalación de dependencias
Instalamos `transformers`, que nos permite cargar modelos como DistilGPT2.

In [1]:
!pip install transformers --quiet

## Paso 2: Cargar el modelo y el tokenizador
Usaremos el modelo `distilgpt2`, una versión liviana de GPT2 que permite generar texto rápidamente.

In [2]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

model_name = 'distilgpt2'
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)
model.eval();

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/762 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/353M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

## Paso 3: Definir un prompt
Este será el inicio de nuestro texto. Puedes cambiarlo para experimentar.

In [32]:
prompt = "Once upon a time, there was a dragon that lived in the mountains and"
input_ids = tokenizer.encode(prompt, return_tensors='pt')
attention_mask = torch.ones_like(input_ids)

## Paso 4: Estrategias de generación
### 1. Greedy Decoding
- El modelo elige el token más probable en cada paso.
- Muy determinista, pero puede generar texto repetitivo o corto.

In [33]:
greedy_output = model.generate(
    input_ids,
    attention_mask=attention_mask,
    max_length=50,
    pad_token_id=tokenizer.eos_token_id
)
print(tokenizer.decode(greedy_output[0], skip_special_tokens=True))

Once upon a time, there was a dragon that lived in the mountains and was a dragon that lived in the mountains and was a dragon that lived in the mountains and was a dragon that lived in the mountains and was a dragon that lived in the mountains


### 2. Beam Search
- Considera múltiples secuencias simultáneamente.
- Mejora la coherencia, pero aún es determinista.
- Beam width define cuántas secuencias se consideran.

In [36]:
beam_output = model.generate(
    input_ids,
    attention_mask=attention_mask,
    num_beams=5,
    max_length=50,
    early_stopping=True,
    pad_token_id=tokenizer.eos_token_id
)
print(tokenizer.decode(beam_output[0], skip_special_tokens=True))

Once upon a time, there was a dragon that lived in the mountains and lived in the mountains.

※
※
※
※
※
※
※


### 3. Top-k Sampling
- En lugar de elegir siempre la opción más probable, selecciona aleatoriamente entre las **k más probables**.
- Introduce diversidad.

In [34]:
topk_output = model.generate(
    input_ids,
    attention_mask=attention_mask,
    do_sample=True,
    top_k=50,
    max_length=50,
    pad_token_id=tokenizer.eos_token_id
)
print(tokenizer.decode(topk_output[0], skip_special_tokens=True))

Once upon a time, there was a dragon that lived in the mountains and had traveled throughout the world in search of an immortal king. He had lived in an ancient city where the Dragon people lived. It was an ancient city where the Dragon people lived


### 4. Top-p (Nucleus) Sampling
- Elige de forma aleatoria dentro del **conjunto más pequeño de palabras cuya probabilidad total supera p**.
- Es más dinámico que top-k.

In [35]:
topp_output = model.generate(
    input_ids,
    attention_mask=attention_mask,
    do_sample=True,
    top_p=0.92,
    top_k=0,
    max_length=50,
    pad_token_id=tokenizer.eos_token_id
)
print(tokenizer.decode(topp_output[0], skip_special_tokens=True))

Once upon a time, there was a dragon that lived in the mountains and ran over to kill all you and its inhabitants.
※Follow Max to popularise or create custom missions, telling the story of what happened to the villagers, assuming you


### 5. Controlando la temperatura
- `temperature < 1.0` hace al modelo más conservador.
- `temperature > 1.0` lo hace más creativo e impredecible.

In [37]:
temperature_output = model.generate(
    input_ids,
    attention_mask=attention_mask,
    do_sample=True,
    temperature=1.5,
    top_k=50,
    max_length=50,
    pad_token_id=tokenizer.eos_token_id
)
print(tokenizer.decode(temperature_output[0], skip_special_tokens=True))

Once upon a time, there was a dragon that lived in the mountains and would probably have died if his life could only been saved: his body. He lived this great creature from eternity. He has a body that no longer existed, and died the


## Paso 5: Comparación de resultados
Analiza con tus alumnos cómo cambia el texto generado dependiendo de la estrategia. ¿Qué diferencias notan en coherencia, fluidez, creatividad o repeticiones?

## Paso 6: Prueba con tu propio prompt

In [40]:
prompt = (
    "Once upon a time, in a land veiled by eternal mist, there was an old dragon "
    "who lived alone in the icy peaks of the northern mountains. Feared by men and forgotten by time, "
    "the dragon spent his days gazing at the stars and writing poems in the snow. "
    "One winter night, a lost child arrived at his cave, shivering and silent. "
    "What happened next would change the fate of both forever. "
    "Write this story in a poetic and emotional style."
)
input_ids = tokenizer.encode(prompt, return_tensors='pt')
attention_mask = torch.ones_like(input_ids)

In [43]:
temperature_output = model.generate(
    input_ids,
    attention_mask=attention_mask,
    do_sample=True,
    temperature=1.5,
    top_k=50,
    max_length=300,
    pad_token_id=tokenizer.eos_token_id
)
print(tokenizer.decode(temperature_output[0], skip_special_tokens=True))

Once upon a time, in a land veiled by eternal mist, there was an old dragon who lived alone in the icy peaks of the northern mountains. Feared by men and forgotten by time, the dragon spent his days gazing at the stars and writing poems in the snow. One winter night, a lost child arrived at his cave, shivering and silent. What happened next would change the fate of both forever. Write this story in a poetic and emotional style.
There was darkness around her so she stood upon a steep hill, one above where two tall men had formed such a body, one above where one was covered in heavy leather on their sides, and a one above which a man could sleep. In one moment the dragon began to lay motionless on the stone pillars beneath the cave at this stage. She made sure that she should never stand in between it and, more often, an other and two. "We are to keep going to bed, we believe." The dragon passed before a wall appeared above the cliff. No body was covered. They stared at her for ten secon

In [47]:
import textwrap

raw_text = ("There was darkness around her so she stood upon a steep hill, one above where two tall men had formed such a body, one above where one was covered in heavy leather on their sides, and a one above which a man could sleep. In one moment the dragon began to lay motionless on the stone pillars beneath the cave at this stage. She made sure that she should never stand in between it and, more often, an other and two. We are to keep going to bed, we believe. The dragon passed before a wall appeared above the cliff. No body was covered. They stared at her for ten second seconds. Suddenly the dragon stepped upon the wall below. He didn't turn back, his throat was cold with cold hunger so it gave way as darkening as he expected, causing darkness in his blood. He then raised the large spider of Death at his eyes with a great deal of fear, his neck twisted, and there was at one point the creature almost struck by him. Im not safe")

# Split por punto y reagrupa cada 2 oraciones como un párrafo
sentences = [s.strip() for s in raw_text.split('.') if s.strip()]
paragraphs = ['. '.join(sentences[i:i+2]) + '.' for i in range(0, len(sentences), 2)]

# Imprime con un salto entre párrafos
for p in paragraphs:
    print(textwrap.fill(p, width=100))
    print()

There was darkness around her so she stood upon a steep hill, one above where two tall men had
formed such a body, one above where one was covered in heavy leather on their sides, and a one above
which a man could sleep. In one moment the dragon began to lay motionless on the stone pillars
beneath the cave at this stage.

She made sure that she should never stand in between it and, more often, an other and two. We are to
keep going to bed, we believe.

The dragon passed before a wall appeared above the cliff. No body was covered.

They stared at her for ten second seconds. Suddenly the dragon stepped upon the wall below.

He didn't turn back, his throat was cold with cold hunger so it gave way as darkening as he
expected, causing darkness in his blood. He then raised the large spider of Death at his eyes with a
great deal of fear, his neck twisted, and there was at one point the creature almost struck by him.

Im not safe.

